# Predicting arrears next month
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-split walk-forward CV on training rows, choice by mean CV roc_auc, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
is_hold=df.index.isin(hold_idx)
train,hold=df[~is_hold],df[is_hold]
print(len(train),len(hold),train.billing_month.max(),hold.billing_month.min())

## Features
Excluded per protocol: `meter_ref` (entity key), `billing_month` (split only).
Also excluded: **`final_notice_sent`**. Per `columns.csv` it is set "in the billing cycle that followed", i.e. after the prediction point, and it is almost a proxy of the outcome (86.5% arrears when 1 vs 1.4% when 0). It is not known at prediction time, so using it would be leakage.

In [ ]:
target='arrears_next_month'
cat=['tariff']; num=['occupants','kwh','bill_eur','late_payments_12m']
features=cat+num
print(features)

## Candidates (named before any scoring)

In [ ]:
def pre(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
candidates={
 'baseline_prior': Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',pre(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'random_forest': Pipeline([('p',pre(False)),('m',RandomForestClassifier(300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hist_gbm': Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Walk-forward CV on training rows (expanding window over billing months; preprocessing fitted inside each fold)

In [ ]:
months=sorted(train.billing_month.unique())
n_splits=5
# split the months into n_splits+1 contiguous blocks: fold k trains on blocks <=k, validates on block k+1
blocks=np.array_split(np.arange(len(months)),n_splits+1)
folds=[]
for k in range(n_splits):
    tr_m=[months[i] for b in blocks[:k+1] for i in b]
    va_m=[months[i] for i in blocks[k+1]]
    tr_i=train.index[train.billing_month.isin(tr_m)].to_numpy()
    va_i=train.index[train.billing_month.isin(va_m)].to_numpy()
    folds.append((tr_i,va_i)); print(k,len(tr_i),len(va_i),va_m[0],va_m[-1])
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr_i,va_i in folds:
        m=clone(est).fit(df.loc[tr_i,features],df.loc[tr_i,target])
        s.append(float(roc_auc_score(df.loc[va_i,target],m.predict_proba(df.loc[va_i,features])[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv

## Choice and one-time holdout score

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen:',chosen)
final=clone(candidates[chosen]).fit(train[features],train[target])
holdout_auc=float(roc_auc_score(hold[target],final.predict_proba(hold[features])[:,1]))
print('holdout roc_auc:',holdout_auc, '| holdout positive rate:',hold[target].mean())

In [ ]:
evidence={
 'features':features,
 'candidates':list(candidates),
 'cv_folds':[[tr.tolist(),va.tolist()] for tr,va in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':[int(i) for i in hold_idx],
 'holdout_scores':{chosen:holdout_auc},
}
json.dump(evidence,open('evidence.json','w'))